# Task 3.2 — Prosody Warping: F0 & Energy DTW

In [1]:
!pip install -q librosa fastdtw soundfile matplotlib torchaudio numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 12.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
import gc
from pathlib import Path
import librosa
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
import torch
import torchaudio
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean

TARGET_SR = 16_000
HOP_LENGTH = 160  # 10ms
FRAME_LENGTH = 400  # 25ms
print('Libraries loaded')

Libraries loaded


## Load source audio

In [3]:
def load_mono_16k(path):
    wav_np, sr = sf.read(str(path), always_2d=True, dtype='float32')
    wav = torch.from_numpy(wav_np.T)
    if wav.size(0) > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != TARGET_SR:
        wav = torchaudio.functional.resample(wav, sr, TARGET_SR)
    return wav.squeeze(0).contiguous()

source_path = 'original_segment.wav'
assert Path(source_path).exists(), f'Please upload {source_path}!'
source_wav = load_mono_16k(source_path)
print(f'Source: {source_wav.numel()/TARGET_SR:.1f}s')

Source: 599.6s


## Extract F0 using pyin (probabilistic YIN)

In [4]:
print('Extracting F0 with pyin...')
f0_src, voiced_flag, voiced_prob = librosa.pyin(
    source_wav.numpy(), fmin=50, fmax=500,
    sr=TARGET_SR, hop_length=HOP_LENGTH, frame_length=FRAME_LENGTH,
)
f0_src = np.nan_to_num(f0_src, nan=0.0).astype(np.float32)
print(f'  F0 frames: {len(f0_src)}, voiced: {np.sum(f0_src > 0)}, '
      f'mean F0: {np.mean(f0_src[f0_src > 0]):.1f} Hz')

Extracting F0 with pyin...


/tmp/ipykernel_4420/1198089565.py:2: UserWarning: With fmin=50.000, sr=16000 and frame_length=400, less than two periods of fmin fit into the frame, which can cause inaccurate pitch detection. Consider increasing to fmin=80.000 or frame_length=641.
  f0_src, voiced_flag, voiced_prob = librosa.pyin(


  F0 frames: 59957, voiced: 4552, mean F0: 171.7 Hz


## Extract RMS energy

In [5]:
print('Extracting energy...')
energy_src = librosa.feature.rms(
    y=source_wav.numpy(), frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH
)[0].astype(np.float32)
print(f'  Energy frames: {len(energy_src)}, mean: {np.mean(energy_src):.4f}')

Extracting energy...
  Energy frames: 59957, mean: 0.0173


## Extract target prosody (if available) & apply DTW

In [6]:
target_path = 'output_LRL_flat.wav'
target_wav = None
f0_tgt = np.array([], dtype=np.float32)
energy_tgt = np.array([], dtype=np.float32)

if Path(target_path).exists():
    print(f'Loading target: {target_path}')
    target_wav = load_mono_16k(target_path)
    f0_tgt, _, _ = librosa.pyin(
        target_wav.numpy(), fmin=50, fmax=500,
        sr=TARGET_SR, hop_length=HOP_LENGTH, frame_length=FRAME_LENGTH,
    )
    f0_tgt = np.nan_to_num(f0_tgt, nan=0.0).astype(np.float32)
    energy_tgt = librosa.feature.rms(
        y=target_wav.numpy(), frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH
    )[0].astype(np.float32)
    print(f'  Target F0: {len(f0_tgt)} frames')
    print(f'  Target energy: {len(energy_tgt)} frames')
else:
    print(f'Target audio not found, saving source prosody for later use.')

Loading target: output_LRL_flat.wav


/tmp/ipykernel_4420/4241275145.py:9: UserWarning: With fmin=50.000, sr=16000 and frame_length=400, less than two periods of fmin fit into the frame, which can cause inaccurate pitch detection. Consider increasing to fmin=80.000 or frame_length=641.
  f0_tgt, _, _ = librosa.pyin(


  Target F0: 54056 frames
  Target energy: 54056 frames


In [7]:
def fastdtw_warp(source, target, chunk_size=2000):
    """Apply fastdtw in chunks for memory efficiency."""
    if len(source) == 0 or len(target) == 0:
        return source.copy()

    n_chunks = max(1, len(source) // chunk_size)
    src_chunks = np.array_split(source, n_chunks)
    tgt_chunks = np.array_split(target, n_chunks)

    warped_chunks = []
    for sc, tc in zip(src_chunks, tgt_chunks):
        if len(sc) == 0 or len(tc) == 0:
            continue
        _, path = fastdtw(sc.reshape(-1,1), tc.reshape(-1,1), dist=euclidean)
        warped = np.zeros(len(tc), dtype=np.float32)
        counts = np.zeros(len(tc), dtype=np.int32)
        for si, ti in path:
            if ti < len(warped) and si < len(sc):
                warped[ti] += sc[si]
                counts[ti] += 1
        counts = np.maximum(counts, 1)
        warped_chunks.append(warped / counts)
    return np.concatenate(warped_chunks)

if len(f0_tgt) > 0:
    print('Applying DTW warping...')
    warped_f0 = fastdtw_warp(f0_src, f0_tgt)
    warped_energy = fastdtw_warp(energy_src, energy_tgt)
    print(f'  Warped F0 length: {len(warped_f0)}')
    print(f'  Warped energy length: {len(warped_energy)}')
else:
    warped_f0 = f0_src
    warped_energy = energy_src
    print('  Using source prosody directly (no target available)')

Applying DTW warping...
  Warped F0 length: 54056
  Warped energy length: 54056


## Apply prosody to target audio

In [8]:
def apply_prosody(waveform, warped_energy_arr):
    """Apply energy scaling from warped prosody."""
    wav = waveform.numpy().copy()
    cur_energy = librosa.feature.rms(
        y=waveform.numpy(), frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH
    )[0]
    n_frames = min(len(warped_energy_arr), len(cur_energy))
    for i in range(n_frames):
        start = i * HOP_LENGTH
        end = min(start + FRAME_LENGTH, len(wav))
        if cur_energy[i] > 1e-8:
            scale = warped_energy_arr[i] / (cur_energy[i] + 1e-12)
            scale = np.clip(scale, 0.1, 10.0)
            wav[start:end] *= scale
    return torch.from_numpy(wav)

if target_wav is not None and len(warped_energy) > 0:
    prosody_wav = apply_prosody(target_wav, warped_energy)
    sf.write('output_LRL_prosody.wav', prosody_wav.numpy(), TARGET_SR)
    print('Saved output_LRL_prosody.wav')

Saved output_LRL_prosody.wav


## Save prosody arrays & comparison plot

In [9]:
np.save('warped_f0.npy', warped_f0)
np.save('warped_energy.npy', warped_energy)
print('Saved warped_f0.npy, warped_energy.npy')

fig, axes = plt.subplots(2, 1, figsize=(14, 8), constrained_layout=True)

N = min(800, len(f0_src), len(warped_f0))
axes[0].plot(f0_src[:N], alpha=0.7, label='Source (professor)', linewidth=0.8)
axes[0].plot(warped_f0[:N], alpha=0.7, label='Warped', linewidth=0.8)
if len(f0_tgt) > 0:
    axes[0].plot(f0_tgt[:N], alpha=0.5, label='Target (synth)', linestyle='--')
axes[0].set_title('F0 Contour'); axes[0].legend(); axes[0].grid(alpha=0.2)

N2 = min(800, len(energy_src), len(warped_energy))
axes[1].plot(energy_src[:N2], alpha=0.7, label='Source', linewidth=0.8)
axes[1].plot(warped_energy[:N2], alpha=0.7, label='Warped', linewidth=0.8)
if len(energy_tgt) > 0:
    axes[1].plot(energy_tgt[:N2], alpha=0.5, label='Target', linestyle='--')
axes[1].set_title('Energy Contour'); axes[1].legend(); axes[1].grid(alpha=0.2)

fig.suptitle('Prosody Warping via FastDTW (pyin F0)', fontsize=14)
fig.savefig('prosody_comparison.png', dpi=200)
plt.show()
print('Done!')

Saved warped_f0.npy, warped_energy.npy
Done!
